In [ ]:
from rio_stac.stac import create_stac_item
from pathlib import Path
from datetime import datetime, timezone
import json
import rasterio
import pystac
import geopandas as gpd 
from shapely.geometry import mapping

In [ ]:
def create_stac_for_raster(input_file, title, description, user_datetime=None):

    print("FUNCTION STARTED (RASTER)")

    input_path = Path(input_file)

    with rasterio.open(input_path) as src:

        print("RASTER LOADED")


        # DATETIME

        # 1. user input OR fallback
        dt = parse_datetime(user_datetime)

        # 2. try raster metadata (optional override if exists)
        raster_dt = None

        if src.tags():
            # common GeoTIFF tag keys (depends on dataset)
            for key in ["TIFFTAG_DATETIME", "datetime", "acquisition_time"]:
                if key in src.tags():
                    try:
                        raster_dt = parse_datetime(src.tags()[key])
                        break
                    except:
                        pass

        if raster_dt:
            dt = raster_dt

        print("DATETIME USED:", dt)

        # -------------------------
        # SPATIAL INFO
        # -------------------------

        epsg = src.crs.to_epsg() if src.crs else None

        left, bottom, right, top = src.bounds
        bbox = [left, bottom, right, top]

        geometry = {
            "type": "Polygon",
            "coordinates": [[
                [left, bottom],
                [right, bottom],
                [right, top],
                [left, top],
                [left, bottom]
            ]]
        }

        # -------------------------
        # STAC ITEM
        # -------------------------

        item = pystac.Item(
            id=input_path.stem,
            geometry=geometry,
            bbox=bbox,
            datetime=dt,
            properties={
                "title": title,
                "description": description,
                "proj:epsg": epsg
            }
        )

        item.add_asset(
            "data",
            pystac.Asset(
                href=input_path.name,
                media_type="image/tiff; application=geotiff",
                roles=["data"],
                title=title,
                description=description
            )
        )

    # -------------------------
    # SAVE JSON
    # -------------------------

    output_json = input_path.with_suffix(".json")

    with open(output_json, "w", encoding="utf-8") as f:
        json.dump(item.to_dict(), f, indent=2, ensure_ascii=False)

    print("STAC METADATA SAVED:")
    print(output_json)

def parse_datetime(dt_input):
    """
    Convert input into timezone-aware UTC datetime.

    Supports:
        - None → returns current UTC time
        - "YYYY-MM-DD" → sets midnight UTC
        - "YYYY-MM-DDTHH:MM:SS" → full datetime
        - datetime object
    """

    if dt_input is None:
        return datetime.now(timezone.utc)

    if isinstance(dt_input, datetime):
        return dt_input.replace(tzinfo=timezone.utc)

    if isinstance(dt_input, str):

        # Date only → midnight UTC
        if len(dt_input) == 10:
            return datetime.strptime(dt_input, "%Y-%m-%d").replace(tzinfo=timezone.utc)

        # Full ISO datetime
        return datetime.fromisoformat(dt_input).replace(tzinfo=timezone.utc)

    raise ValueError("Unsupported datetime format")

In [ ]:
def create_stac_for_vector(input_file, title, description): 
    """ 
    Create STAC metadata for GPKG or SHP files. 
    Inputs: - input_file : path to .gpkg or .shp - title - description 
    Výstup: - .json near the input file
    """ 
    input_path = Path(input_file) 
    
    # load data 
    gdf = gpd.read_file(input_path) 
    
    # CRS 
    epsg = gdf.crs.to_epsg() if gdf.crs else None 
    
    # bbox in original CRS 
    bounds = gdf.total_bounds.tolist() 
    
    # geometry
    union_geom = gdf.geometry.union_all() 
    
    geometry = mapping(union_geom) 
    
    # attributes and data types 
    columns = [] 
    
    for col, dtype in gdf.dtypes.items():  
        
        if col == gdf.geometry.name: 
            continue 
        
        columns.append({ 
            "name": col, 
            "type": str(dtype) 
        }) 
        
    # media type 
    suffix = input_path.suffix.lower() 
        
    if suffix == ".gpkg": 
        media_type = "application/geopackage+sqlite3" 
        
        
    elif suffix == ".shp": 
        media_type = "application/x-esri-shapefile" 
        
        
    else: 
        raise ValueError("Only .gpkg or .shp") 
            
    # STAC item 
        
    item = pystac.Item( 
        id=input_path.stem, 
        geometry=geometry, 
        bbox=bounds, 
        datetime=datetime.utcnow(), 
        properties={ 
            "title": title, 
            "description": description, 
            "proj:epsg": epsg 
        } 
    ) 
        
    # asset 
    item.add_asset( 
        "data", 
        pystac.Asset( 
            href=input_path.name, 
            media_type=media_type, 
            roles=["data"], 
            title=title, 
            description=description, 
            extra_fields={ 
                "table:columns": columns 
            } 
        ) 
    ) 
        
    # output .json flóle
    output_json = input_path.with_suffix(".json") 
        
    with open(output_json, "w", encoding="utf-8") as f: 
        json.dump(item.to_dict(), f, indent=2, ensure_ascii=False) 
            
    print(f"STAC metadata saved:") 
    print(output_json)




In [ ]:
# VECTOR gpkg, shp

create_stac_for_vector( 
    input_file =  r"f:\PhD\Projekty\GAIA\sites\AMD_CZE_CSA_Most\static\AOI_CSA_Most.gpkg", 
    title =       "AOI CSA Most", 
    description = "Area of interest for locality CSA Most, Czechia." 
)

In [ ]:
# RASTER dataset

create_stac_for_raster(
    input_file =   r"f:\PhD\Projekty\GAIA\sites\STAC test\A_2024-01-19_Yxsjoberg_S2_SR_HarmL2A_T33VVG_A035884_20240119T103252 – kópia.tif",
    title =        "Sentinel-2 Yxsjoberg",
    description =  "Sentinel-2 raster dataset over Yxsjoberg area",
    user_datetime= None #None for actual time or "2026-02-10T13:45:00" or "2026-02-10"
)

# or full datetime
# user_datetime="2026-02-10T13:45:00"

In [ ]:
# FOR AMD
name = "Kitwe"
parent_folder_name = "ZMB_Kitwe"
type_data = "AMD"
country = "Zambia"



create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\AOI_{name}.gpkg",
    title= f"Area of Interest – {name}",
    description= f"Polygon defining the Area of Interest (AOI) for the {type_data} {name} study site, {country}. Used as the spatial boundary for analysis and processing."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Mask_Clean_water_{name}_Polygon.gpkg",
    title= f"Clean Water Mask – {name}",
    description="Manually digitized polygon representing areas of confirmed clean water within the {name} site."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Mask_TSF_{name}_Polygon.gpkg",
    title= f"TSF Water Mask – {name}",
    description="Manually digitized polygon representing TSF-affected water areas within the {name} site."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Reference_Clean_water_{name}_Point.gpkg",
    title= f"Reference Points – Clean Water ({name})",
    description="Point dataset representing validation locations placed inside manually digitized clean water polygons. Used as reference samples for verification of classification accuracy."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Reference_TSF_{name}_Point.gpkg",
    title= f"Reference Points – TSF Water ({name})",
    description="Point dataset representing validation locations placed inside TSF-affected water polygons. Used as reference samples for validation and quality control."
)

In [ ]:
#FOR SLOPE

name = "TD5_Western_Platinum"
parent_folder_name = "SAF_TD5_Western_Platinum"
type_data = "Slope"
country = "South Africa"


create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\AOI_{name}.gpkg",
    title= f"Area of Interest – {name}",
    description= f"Polygon defining the Area of Interest (AOI) for the {type_data} {name} study site, {country}. Used as the spatial boundary for analysis and processing."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Mask_Stable_Slope_{name}_Polygon.gpkg",
    title= f"Clean Water Mask – {name}",
    description="Manually digitized polygon representing areas of confirmed stable slope around the {name} site."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Mask_TSF_Slope_{name}_Polygon.gpkg",
    title= f"TSF Water Mask – {name}",
    description="Manually digitized polygon representing TSF-affected unstable slope areas within the {name} site."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Reference_Stable_Slope_{name}_Point.gpkg",
    title= f"Reference Points – Clean Water ({name})",
    description="Point dataset representing validation locations placed inside manually digitized stable slope polygons. Used as reference samples for verification of classification accuracy."
)

create_stac_for_vector(
    input_file=rf"f:\PhD\Projekty\GAIA\sites\{type_data}_{parent_folder_name}\static\Reference_TSF_Slope_{name}_Point.gpkg",
    title= f"Reference Points – TSF Water ({name})",
    description="Point dataset representing validation locations placed inside TSF-affected unstable slope polygons. Used as reference samples for validation and quality control."
)